<a href="https://colab.research.google.com/github/MUHAMMAD-IJLAL-061/rag-document-verifier/blob/main/Rag_document_verifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pymupdf easyocr sentence-transformers faiss-cpu gradio google-genai pandas

In [ ]:
import os, re, json, glob, time, unicodedata
import numpy as np, pandas as pd, torch
from google.colab import drive, userdata

drive.mount('/content/drive')

BASE      = '/content/drive/MyDrive/viva_mentor_rag'
PDF_DIR   = f'{BASE}/pdfs'
INDEX_DIR = f'{BASE}/index_en'      # NEW folder: English-only index
EVAL_DIR  = f'{BASE}/eval'
for d in (PDF_DIR, INDEX_DIR, EVAL_DIR):
    os.makedirs(d, exist_ok=True)

DEFAULT_PDFS = ['english.pdf']      # PDFs indexed at start. Add more English PDFs here.
THRESHOLD = 0.30                    # best reranker score below this -> "not found"
MAX_PAGES = 40                      # pages read from an uploaded PDF in the demo
NOT_FOUND = "Not found in the document."

print('GPU available:', torch.cuda.is_available())
print('PDFs on Drive:', os.listdir(PDF_DIR))
missing = [f for f in DEFAULT_PDFS if not os.path.exists(f'{PDF_DIR}/{f}')]
if missing: print('WARNING: missing PDFs:', missing)

In [ ]:
from google import genai
from google.genai import types

CLIENTS = [genai.Client(api_key=userdata.get('GEMINI_API_KEY'))]
try:
    CLIENTS.append(genai.Client(api_key=userdata.get('GEMINI_API_KEY_2')))
    print('Second key loaded.')
except Exception:
    print('No second key (optional).')

BAD = ('image','tts','live','audio','embed','robotics','computer','imagen','veo','learnlm','gemma','omni')

def flash_models(cl):
    """Ask Google which text 'flash' models this key can use today."""
    names = []
    for m in cl.models.list():
        n = m.name.replace('models/', '')
        acts = getattr(m, 'supported_actions', None) or []
        if 'flash' in n and 'generateContent' in acts and not any(b in n for b in BAD):
            names.append(n)
    return sorted(names, key=lambda n: 'lite' in n)

CHAIN = [(cl, n) for cl in CLIENTS for n in flash_models(cl)]
print('Models in fallback chain:', len(CHAIN))

def llm_generate(prompt):
    """Try every model in the chain until one answers."""
    last = None
    for rnd in range(2):
        for i, (cl, model) in enumerate(CHAIN):
            try:
                r = cl.models.generate_content(
                    model=model, contents=prompt,
                    config=types.GenerateContentConfig(temperature=0))
                if r.text:
                    if i > 0: CHAIN.insert(0, CHAIN.pop(i))
                    return r.text.strip()
            except Exception as e:
                last = e
        time.sleep(10)
    raise RuntimeError(f'All models failed. Last error: {str(last)[:150]}')

print('Gemini test:', llm_generate('Say OK'))

In [ ]:
import pymupdf, easyocr

def normalize_text(t):
    t = unicodedata.normalize('NFKC', t)
    t = re.sub(r'[ \t]+', ' ', t)
    t = re.sub(r'\n{3,}', '\n\n', t)
    return t.strip()

def looks_bad(raw):
    """True if the page text is empty or broken -> use OCR."""
    return len(raw.strip()) < 30 or raw.count('\ufffd') > 3

ocr_reader = easyocr.Reader(['en'], gpu=True)          # English OCR for scanned pages

def ocr_page(page, zoom=2):
    pix = page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom))
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
    if pix.n == 4: img = img[:, :, :3]
    return '\n'.join(ocr_reader.readtext(img, detail=0, paragraph=True))

def read_pdf(pdf_path, max_pages=None, progress=None):
    """PDF -> list of {file, page, text, method}. Page numbers start at 1."""
    doc = pymupdf.open(pdf_path)
    n = min(len(doc), max_pages) if max_pages else len(doc)
    name, pages = os.path.basename(pdf_path), []
    for i in range(n):
        raw = doc[i].get_text()
        text, method = (ocr_page(doc[i]), 'ocr') if looks_bad(raw) else (raw, 'text')
        text = normalize_text(text)
        if text:
            pages.append({'file': name, 'page': i + 1, 'text': text, 'method': method})
        if progress: progress((i + 1) / n, desc=f'Reading page {i + 1} of {n}')
    return pages

def split_sentences(text):
    parts = re.split(r'(?<=[.!?])\s+|\n+', text)
    return [p.strip() for p in parts if p.strip()]

def chunk_pages(pages, chunk_size=600, overlap=100):
    """Pack sentences into ~chunk_size characters; chunks never cross a page."""
    chunks = []
    for pg in pages:
        n, current, new_count = 0, [], 0
        def flush():
            nonlocal n, current, new_count
            if new_count > 0:
                n += 1
                chunks.append({'chunk_id': f"{pg['file']}_p{pg['page']}_c{n}", 'file': pg['file'],
                               'page': pg['page'], 'method': pg['method'], 'text': ' '.join(current)})
            keep, total = [], 0
            for s in reversed(current):
                if total + len(s) > overlap: break
                keep.insert(0, s); total += len(s)
            current, new_count = keep, 0
        for s in split_sentences(pg['text']):
            if len(s) > chunk_size:
                flush(); current = []
                for i in range(0, len(s), chunk_size):
                    current, new_count = [s[i:i + chunk_size]], 1
                    flush(); current = []
                continue
            if sum(len(x) + 1 for x in current) + len(s) > chunk_size and new_count > 0:
                flush()
            current.append(s); new_count += 1
        flush()
    return chunks
print('Reading + chunking functions ready.')

## Cell 5: Embedding model, reranker, FAISS (first run downloads about 4 GB)

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

embedder = SentenceTransformer('BAAI/bge-m3', device='cuda')
embedder.half()
embedder.max_seq_length = 512
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', device='cuda', max_length=512)
reranker.model.half()

def build_faiss(chunks):
    vecs = embedder.encode([c['text'] for c in chunks], batch_size=16,
                           normalize_embeddings=True, show_progress_bar=False)
    vecs = np.asarray(vecs, dtype='float32')
    index = faiss.IndexFlatIP(vecs.shape[1])
    index.add(vecs)
    return index

def save_index(index, chunks, pages=None, folder=INDEX_DIR):
    faiss.write_index(index, f'{folder}/faiss.index')
    with open(f'{folder}/chunks.json', 'w', encoding='utf-8') as f: json.dump(chunks, f, ensure_ascii=False)
    if pages is not None:
        with open(f'{folder}/pages.json', 'w', encoding='utf-8') as f: json.dump(pages, f, ensure_ascii=False)

def load_index(folder=INDEX_DIR):
    index = faiss.read_index(f'{folder}/faiss.index')
    with open(f'{folder}/chunks.json', encoding='utf-8') as f: chunks = json.load(f)
    with open(f'{folder}/pages.json', encoding='utf-8') as f: pages = json.load(f)
    return index, chunks, pages
print('Models loaded. Vector size:', embedder.get_sentence_embedding_dimension())

## Cell 6: Retrieval and `ask_rag`

In [ ]:
SYSTEM_PROMPT = """You answer questions using ONLY the numbered context chunks below.
Rules:
1. Use only facts that appear in the chunks. Never use outside knowledge.
2. If the chunks do not clearly contain the answer, reply exactly: NOT_FOUND
3. Answer in English, in a short and clear way.
4. After each fact write its source like (Page 3, file.pdf).
5. Some chunks may come from OCR and have spelling mistakes. Use the meaning and fix obvious typos."""

def search(question, k=5):
    q = embedder.encode([question], normalize_embeddings=True)
    scores, ids = INDEX.search(np.asarray(q, dtype='float32'), min(k, INDEX.ntotal))
    return [(float(s), CHUNKS[i]) for s, i in zip(scores[0], ids[0]) if i >= 0]

def retrieve(question, k_first=20, k_final=5, use_reranker=True):
    """FAISS finds k_first candidates; the reranker keeps the best k_final."""
    cands = search(question, k=k_first)
    if not use_reranker:
        return cands[:k_final]
    scores = reranker.predict([(question, c['text']) for _, c in cands], batch_size=16,
                              activation_fn=torch.nn.Sigmoid())
    ranked = sorted(zip(map(float, scores), [c for _, c in cands]), key=lambda x: x[0], reverse=True)
    return ranked[:k_final]

def ask_rag(question, k_final=5, threshold=None, use_reranker=True):
    """Returns {'answer': str, 'sources': [...], 'top_score': float}"""
    if threshold is None: threshold = THRESHOLD
    question = (question or '').strip()
    if not question:
        return {'answer': 'Please type a question.', 'sources': [], 'top_score': 0}
    if re.search(r'[\u0600-\u06FF]', question):            # Urdu letters found
        return {'answer': 'Only English is supported in this version. Please ask in English.',
                'sources': [], 'top_score': 0}
    results = retrieve(question, k_final=k_final, use_reranker=use_reranker)
    top_score = results[0][0]
    if use_reranker and top_score < threshold:              # safety net 1
        return {'answer': NOT_FOUND, 'sources': [], 'top_score': top_score}
    good = [(s, c) for s, c in results if (s >= threshold or not use_reranker)]
    context = '\n\n'.join(f"[Chunk {i+1} | file: {c['file']} | page: {c['page']}]\n{c['text']}"
                           for i, (s, c) in enumerate(good))
    prompt = f"{SYSTEM_PROMPT}\n\nCONTEXT:\n{context}\n\nQUESTION: {question}\n\nANSWER:"
    try:
        answer = llm_generate(prompt)
    except Exception as e:
        return {'answer': f'AI service error: {e}', 'sources': [], 'top_score': top_score}
    if 'NOT_FOUND' in answer:                               # safety net 2
        return {'answer': NOT_FOUND, 'sources': [], 'top_score': top_score}
    sources = [{'file': c['file'], 'page': c['page'], 'score': round(s, 3), 'snippet': c['text'][:200]}
               for s, c in good]
    return {'answer': answer, 'sources': sources, 'top_score': top_score}
print('RAG functions ready.')

In [ ]:
INDEX, CHUNKS, ALL_PAGES, INDEXED_FILE = None, None, None, None

def build_index(pdf_path, max_pages=MAX_PAGES, save=False, progress=None):
    """Read a PDF, chunk it, embed it and make it the active index."""
    global INDEX, CHUNKS, ALL_PAGES, INDEXED_FILE
    doc_len = len(pymupdf.open(pdf_path))
    pages = read_pdf(pdf_path, max_pages=max_pages, progress=progress)
    if not pages: raise ValueError('No readable text found in this PDF.')
    if progress: progress(0.95, desc='Building search index...')
    chunks = chunk_pages(pages)
    INDEX, CHUNKS, ALL_PAGES, INDEXED_FILE = build_faiss(chunks), chunks, pages, os.path.basename(pdf_path)
    if save: save_index(INDEX, CHUNKS, pages)
    return {'file': INDEXED_FILE, 'pages': len(pages), 'chunks': len(chunks),
            'ocr_pages': sum(p['method'] == 'ocr' for p in pages), 'truncated': doc_len > max_pages}

def load_default_index(rebuild=False):
    """Index the PDFs listed in DEFAULT_PDFS. Loads from Drive if already saved."""
    global INDEX, CHUNKS, ALL_PAGES, INDEXED_FILE
    if not rebuild and os.path.exists(f'{INDEX_DIR}/faiss.index') and os.path.exists(f'{INDEX_DIR}/pages.json'):
        INDEX, CHUNKS, ALL_PAGES = load_index()
        print('Loaded saved index from Drive.')
    else:
        pages = []
        for name in DEFAULT_PDFS:
            pages += read_pdf(f'{PDF_DIR}/{name}')
        CHUNKS = chunk_pages(pages); ALL_PAGES = pages
        INDEX = build_faiss(CHUNKS)
        save_index(INDEX, CHUNKS, pages)
        print('Built and saved a new index.')
    INDEXED_FILE = ', '.join(DEFAULT_PDFS)
    print('Chunks in index:', INDEX.ntotal)

load_default_index()     # use load_default_index(rebuild=True) if you change DEFAULT_PDFS

In [ ]:
for q in ['Who founded Wikipedia?', 'Which organization owns Wikipedia?', 'What is the capital of France?']:
    r = ask_rag(q)
    print('=' * 60, '\nQ:', q, '\nScore:', round(r['top_score'], 3), '\nA:', r['answer'])

In [ ]:
import gradio as gr

def index_ui(file, progress=gr.Progress()):
    if file is None: return 'Please upload a PDF first.'
    path = getattr(file, 'name', file)
    try:
        info = build_index(path, progress=progress)
    except Exception as e:
        return f'Could not read this PDF: {e}'
    msg = (f"✅ Indexed **{info['file']}**: {info['pages']} pages, {info['chunks']} chunks "
           f"({info['ocr_pages']} pages read with OCR).")
    if info['truncated']: msg += f' ⚠️ Only the first {MAX_PAGES} pages were used.'
    return msg

def ask_ui(question):
    if INDEXED_FILE is None: return 'Please upload and index a PDF first.', ''
    if not question or not question.strip(): return 'Please type a question.', ''
    try:
        r = ask_rag(question)
    except Exception as e:
        return f'Something went wrong: {e}', ''
    seen, lines = set(), []
    for s in r['sources']:
        key = (s['file'], s['page'])
        if key in seen: continue
        seen.add(key)
        lines.append(f"**{s['file']} - Page {s['page']}** (relevance {s['score']}): {s['snippet']}...")
    return r['answer'], '\n\n'.join(lines) or '_No sources (answer not found)._'

with gr.Blocks(title='English RAG') as demo:
    gr.Markdown('# Ask questions about your PDF (English)\n'
                'Upload a PDF, wait for "Indexed", then ask your question in English. '
                '(Until you upload, the built-in test PDF is searched.)')
    pdf = gr.File(label='Upload PDF', file_types=['.pdf'])
    status = gr.Markdown()
    q = gr.Textbox(label='Your question', placeholder='Ask in English')
    btn = gr.Button('Ask', variant='primary')
    ans = gr.Markdown(label='Answer')
    src = gr.Markdown(label='Sources')
    pdf.change(index_ui, pdf, status)
    btn.click(ask_ui, q, [ans, src])
    q.submit(ask_ui, q, [ans, src])

demo.launch(share=True, debug=True)     # on demo day you may use debug=False

In [ ]:
load_default_index()

# Evaluatiion
EVAL = [
 {'q':'Who founded Wikipedia?','files':['english.pdf'],'find':'Larry Sanger','keywords':['Jimmy Wales','Larry Sanger']},
 {'q':'Which organization owns Wikipedia?','files':['english.pdf'],'find':'Wikimedia Foundation','keywords':['Wikimedia']},
 {'q':'What type of website is Wikipedia?','files':['english.pdf'],'find':'Online encyclopedia','keywords':['encyclopedia']},
 {'q':'Which Creative Commons license covers most Wikipedia text?','files':['english.pdf'],'find':'Share-Alike','keywords':['Share-Alike','ShareAlike','4.0']},
 {'q':'Which anniversary does the modified Wikipedia logo celebrate?','files':['english.pdf'],'find':'25th anniversary','keywords':['25th','25']},
 # ADD 10+ MORE from pages 2-11 of english.pdf (copy a phrase from the page into 'find').
 {'q':'What is the capital of France?','answerable':False,'absent':['paris']},
 {'q':'Who is the CEO of Google?','answerable':False,'absent':['pichai']},
 {'q':'What is the chemical formula of water?','answerable':False,'absent':['h2o']},
 {'q':'What is the salary of Jimmy Wales?','answerable':False,'absent':['salary']},
]
norm = lambda s: normalize_text(s).lower()
for e in EVAL:
    e.setdefault('answerable', True)
    if e['answerable']:
        key = norm(e['find'])
        e['pages'] = sorted({p['page'] for p in ALL_PAGES if p['file'] in e['files'] and key in norm(p['text'])})
        if not e['pages']: print('NOT FOUND in PDF, fix this question:', e['q'])
    else:
        for w in e.get('absent', []):
            if any(w in p['text'].lower() for p in ALL_PAGES):
                print(f"WARNING: '{w}' appears in the PDF, so this question may be answerable:", e['q'])
print('Questions:', len(EVAL), '| answerable:', sum(e['answerable'] for e in EVAL))

def hit(item, res): return any(c['file'] in item['files'] and c['page'] in item['pages'] for _, c in res)

def eval_retrieval(use_reranker):
    h1 = h5 = n = 0
    for it in EVAL:
        if not it['answerable']: continue
        res = retrieve(it['q'], k_final=5, use_reranker=use_reranker); n += 1
        h1 += hit(it, res[:1]); h5 += hit(it, res)
    return {'reranker': use_reranker, 'questions': n, 'recall@1': round(h1/n, 2), 'recall@5': round(h5/n, 2)}

def eval_answers():
    rows = []
    for it in EVAL:
        r = ask_rag(it['q']); refused = (r['answer'] == NOT_FOUND)
        err = r['answer'].startswith('AI service error')
        ok = ((not refused) and (not err) and any(k.lower() in r['answer'].lower() for k in it['keywords'])) \
             if it['answerable'] else refused
        rows.append({'q': it['q'], 'answerable': it['answerable'], 'correct': ok, 'api_error': err,
                     'score': round(r['top_score'], 3), 'answer': r['answer'][:120]})
        time.sleep(2)
    return pd.DataFrame(rows)

print(pd.DataFrame([eval_retrieval(False), eval_retrieval(True)]).to_string(index=False))
df = eval_answers()
print('\nAPI errors (rerun if > 0):', int(df.api_error.sum()))
print('Answer accuracy   :', round(df[df.answerable].correct.mean(), 2))
print('Not-found accuracy:', round(df[~df.answerable].correct.mean(), 2))
print('\nWrong answers:'); print(df[~df.correct][['q', 'score', 'answer']].to_string())
print('\nLowest answerable score:', df[df.answerable].score.min(), '| Highest unanswerable score:', df[~df.answerable].score.max())
df.to_csv(f'{EVAL_DIR}/results_en.csv', index=False)